In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df = pd.read_csv(
    "../data/weather_cleaned.csv",
    parse_dates=["datetime"]
)

In [19]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    shuffle=False
)

In [21]:
features = [
    'T2M', 'RH2M', 'PS',
    'hour', 'month',

    'T2M_diff', 'RH2M_diff', 'PS_diff',

    'T2M_roll_mean', 'RH2M_roll_mean', 'PS_roll_mean',

    'T2M_roll_std', 'RH2M_roll_std', 'PS_roll_std',

    'T2M_roll_dev', 'RH2M_roll_dev', 'PS_roll_dev'
]


In [22]:
train_df = train_df.copy()

# Time features
train_df['hour'] = train_df['datetime'].dt.hour
train_df['month'] = train_df['datetime'].dt.month

# Change from previous hour
for col in ['T2M', 'RH2M', 'PS']:
    train_df[f'{col}_diff'] = train_df[col].diff()

    # Previous 24-hour behaviour
    train_df[f'{col}_roll_mean'] = train_df[col].rolling(24).mean()
    train_df[f'{col}_roll_std'] = train_df[col].rolling(24).std()

    # Difference from recent average
    train_df[f'{col}_roll_dev'] = (
        train_df[col] - train_df[f'{col}_roll_mean']
    )

In [23]:
train_df = train_df.dropna().reset_index(drop=True)

In [24]:
print(train_df.shape)
train_df.head()

(25455, 18)


,T2M,RH2M,PS,datetime,hour,month,T2M_diff,T2M_roll_mean,T2M_roll_std,T2M_roll_dev,RH2M_diff,RH2M_roll_mean,RH2M_roll_std,RH2M_roll_dev,PS_diff,PS_roll_mean,PS_roll_std,PS_roll_dev
0,11.78,48.78,996.7,2023-01-01 23:00:00,23,1,-0.78,12.904167,5.030627,-1.124167,2.77,49.895833,14.846591,-1.115833,0.0,995.862500,0.920214,0.837500
1,10.41,54.30,996.5,2023-01-02 00:00:00,0,1,-1.37,13.021250,4.933574,-2.611250,5.52,49.382083,14.450394,4.917917,-0.2,995.920833,0.914130,0.579167
2,9.59,58.03,996.3,2023-01-02 01:00:00,1,1,-0.82,13.128750,4.823784,-3.538750,3.73,48.949583,14.004587,9.080417,-0.2,995.979167,0.890520,0.320833
3,9.07,60.90,995.9,2023-01-02 02:00:00,2,1,-0.52,13.217083,4.724100,-4.147083,2.87,48.625000,13.611943,12.275000,-0.4,996.020833,0.860727,-0.120833
4,8.54,63.80,995.4,2023-01-02 03:00:00,3,1,-0.53,13.310000,4.602711,-4.770000,2.90,48.334583,13.186637,15.465417,-0.5,996.054167,0.817727,-0.654167


In [26]:
df_anomaly = test_df.copy()

df_anomaly['is_anomaly'] = 0
df_anomaly['anomaly_type'] = 'normal'

In [27]:
np.random.seed(42)

spike_idx = np.random.choice(
    df_anomaly.index,
    size=100,
    replace=False
)

df_anomaly.loc[spike_idx, 'T2M'] += np.random.uniform(
    15, 30, size=100
)

df_anomaly.loc[spike_idx, 'is_anomaly'] = 1
df_anomaly.loc[spike_idx, 'anomaly_type'] = 'temperature_spike'

In [28]:
df_anomaly[df_anomaly['anomaly_type'] == 'temperature_spike'].head()

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type
25501,46.980427,29.57,992.0,2025-11-28 13:00:00,1,temperature_spike
25509,45.271093,58.07,992.3,2025-11-28 21:00:00,1,temperature_spike
25574,45.772884,21.31,988.9,2025-12-01 14:00:00,1,temperature_spike
25610,30.777043,62.29,991.8,2025-12-03 02:00:00,1,temperature_spike
25695,50.220635,30.54,992.5,2025-12-06 15:00:00,1,temperature_spike


In [29]:
freeze_starts = np.random.choice(
    df_anomaly.index[:-12],
    size=10,
    replace=False
)

for start in freeze_starts:
    value = df_anomaly.loc[start, 'T2M']

    df_anomaly.loc[start:start+11, 'T2M'] = value
    df_anomaly.loc[start:start+11, 'is_anomaly'] = 1
    df_anomaly.loc[start:start+11, 'anomaly_type'] = 'temperature_frozen'

In [30]:
start = freeze_starts[0]

df_anomaly.loc[
    start-3:start+14,
    ['datetime', 'T2M', 'is_anomaly', 'anomaly_type']
]

,datetime,T2M,is_anomaly,anomaly_type
26319,2026-01-01 15:00:00,19.56,0,normal
26320,2026-01-01 16:00:00,18.31,0,normal
26321,2026-01-01 17:00:00,17.16,0,normal
26322,2026-01-01 18:00:00,16.18,1,temperature_frozen
26323,2026-01-01 19:00:00,16.18,1,temperature_frozen
26324,2026-01-01 20:00:00,16.18,1,temperature_frozen
26325,2026-01-01 21:00:00,16.18,1,temperature_frozen
26326,2026-01-01 22:00:00,16.18,1,temperature_frozen
26327,2026-01-01 23:00:00,16.18,1,temperature_frozen
26328,2026-01-02 00:00:00,16.18,1,temperature_frozen


In [31]:
drift_starts = np.random.choice(
    df_anomaly.index[:-24],
    size=10,
    replace=False
)

for start in drift_starts:
    idx = range(start, start + 24)

    # Error gradually grows from 0 to +10°C
    drift = np.linspace(0, 10, 24)

    df_anomaly.loc[idx, 'T2M'] += drift
    df_anomaly.loc[idx, 'is_anomaly'] = 1
    df_anomaly.loc[idx, 'anomaly_type'] = 'temperature_drift'

In [32]:
start = drift_starts[0]

df_anomaly.loc[
    start:start+23,
    ['datetime', 'T2M', 'is_anomaly', 'anomaly_type']
]

,datetime,T2M,is_anomaly,anomaly_type
31145,2026-07-21 17:00:00,31.590000,1,temperature_drift
31146,2026-07-21 18:00:00,30.874783,1,temperature_drift
31147,2026-07-21 19:00:00,30.029565,1,temperature_drift
31148,2026-07-21 20:00:00,29.654348,1,temperature_drift
31149,2026-07-21 21:00:00,29.559130,1,temperature_drift
31150,2026-07-21 22:00:00,29.573913,1,temperature_drift
31151,2026-07-21 23:00:00,29.538696,1,temperature_drift
31152,2026-07-22 00:00:00,29.413478,1,temperature_drift
31153,2026-07-22 01:00:00,29.508261,1,temperature_drift
31154,2026-07-22 02:00:00,29.823043,1,temperature_drift


In [33]:
missing_idx = np.random.choice(
    df_anomaly.index,
    size=100,
    replace=False
)

df_anomaly.loc[missing_idx, 'T2M'] = np.nan
df_anomaly.loc[missing_idx, 'is_anomaly'] = 1
df_anomaly.loc[missing_idx, 'anomaly_type'] = 'communication_error'

In [34]:
multi_idx = np.random.choice(
    df_anomaly.dropna().index,
    size=100,
    replace=False
)

source_idx = np.random.choice(
    df.index,
    size=100,
    replace=False
)

# Keep temperature, but replace humidity and pressure
df_anomaly.loc[multi_idx, 'RH2M'] = df.loc[source_idx, 'RH2M'].values
df_anomaly.loc[multi_idx, 'PS'] = df.loc[source_idx, 'PS'].values

df_anomaly.loc[multi_idx, 'is_anomaly'] = 1
df_anomaly.loc[multi_idx, 'anomaly_type'] = 'multivariate_inconsistency'

In [35]:
df_anomaly['hour'] = df_anomaly['datetime'].dt.hour
df_anomaly['month'] = df_anomaly['datetime'].dt.month

for col in ['T2M', 'RH2M', 'PS']:
    df_anomaly[f'{col}_diff'] = df_anomaly[col].diff()

    df_anomaly[f'{col}_roll_mean'] = (
        df_anomaly[col].rolling(24).mean()
    )

    df_anomaly[f'{col}_roll_std'] = (
        df_anomaly[col].rolling(24).std()
    )

    df_anomaly[f'{col}_roll_dev'] = (
        df_anomaly[col] - df_anomaly[f'{col}_roll_mean']
    )

In [36]:
df_anomaly.head()

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type,hour,month,T2M_diff,T2M_roll_mean,T2M_roll_std,T2M_roll_dev,RH2M_diff,RH2M_roll_mean,RH2M_roll_std,RH2M_roll_dev,PS_diff,PS_roll_mean,PS_roll_std,PS_roll_dev
25478,24.73,25.67,993.2,2025-11-27 14:00:00,0,normal,14,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25479,23.90,29.49,993.0,2025-11-27 15:00:00,0,normal,15,11,-0.83,NaN,NaN,NaN,3.82,NaN,NaN,NaN,-0.2,NaN,NaN,NaN
25480,22.10,36.10,993.1,2025-11-27 16:00:00,0,normal,16,11,-1.80,NaN,NaN,NaN,6.61,NaN,NaN,NaN,0.1,NaN,NaN,NaN
25481,21.01,32.80,993.3,2025-11-27 17:00:00,0,normal,17,11,-1.09,NaN,NaN,NaN,-3.30,NaN,NaN,NaN,0.2,NaN,NaN,NaN
25482,20.33,33.69,993.9,2025-11-27 18:00:00,0,normal,18,11,-0.68,NaN,NaN,NaN,0.89,NaN,NaN,NaN,0.6,NaN,NaN,NaN


In [37]:
print(df_anomaly['anomaly_type'].value_counts())

print("\nAnomaly labels:")
print(df_anomaly['is_anomaly'].value_counts())

print("\nMissing values:")
print(df_anomaly[['T2M', 'RH2M', 'PS']].isna().sum())

anomaly_type
normal                        5733
temperature_drift              227
temperature_frozen             119
multivariate_inconsistency     100
communication_error            100
temperature_spike               91
Name: count, dtype: int64

Anomaly labels:
is_anomaly
0    5733
1     637
Name: count, dtype: int64

Missing values:
T2M     100
RH2M      0
PS        0
dtype: int64


In [38]:
df_anomaly.isna().sum()

T2M                100
RH2M                 0
PS                   0
datetime             0
is_anomaly           0
anomaly_type         0
hour                 0
month                0
T2M_diff           200
T2M_roll_mean     2088
T2M_roll_std      2088
T2M_roll_dev      2088
RH2M_diff            1
RH2M_roll_mean      23
RH2M_roll_std       23
RH2M_roll_dev       23
PS_diff              1
PS_roll_mean        23
PS_roll_std         23
PS_roll_dev         23
dtype: int64

In [39]:
communication_errors = df_anomaly[
    df_anomaly['T2M'].isna()
].copy()

In [40]:
features = [
    'T2M', 'RH2M', 'PS',
    'hour', 'month',
    'T2M_diff', 'RH2M_diff', 'PS_diff',
    'T2M_roll_mean', 'RH2M_roll_mean', 'PS_roll_mean',
    'T2M_roll_std', 'RH2M_roll_std', 'PS_roll_std',
    'T2M_roll_dev', 'RH2M_roll_dev', 'PS_roll_dev'
]

test_ml = df_anomaly.dropna(subset=features).copy()

In [41]:
test_ml

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type,hour,month,T2M_diff,T2M_roll_mean,T2M_roll_std,T2M_roll_dev,RH2M_diff,RH2M_roll_mean,RH2M_roll_std,RH2M_roll_dev,PS_diff,PS_roll_mean,PS_roll_std,PS_roll_dev
25501,46.980427,29.57,992.0,2025-11-28 13:00:00,1,temperature_spike,13,11,21.640427,18.821684,7.666723,28.158743,-1.25,45.921250,15.551848,-16.351250,14.2,993.350000,3.409705,-1.350000
25502,25.400000,30.23,991.3,2025-11-28 14:00:00,0,normal,14,11,-21.580427,18.849601,7.690356,6.550399,0.66,46.111250,15.319801,-15.881250,-0.7,993.270833,3.435300,-1.970833
25503,24.330000,34.76,991.1,2025-11-28 15:00:00,0,normal,15,11,-1.070000,18.867518,7.703124,5.462482,4.53,46.330833,15.107503,-11.570833,-0.2,993.191667,3.463589,-2.091667
25504,20.900000,52.26,991.2,2025-11-28 16:00:00,0,normal,16,11,-3.430000,18.817518,7.685103,2.082482,17.50,47.004167,14.991370,5.255833,0.1,993.112500,3.487407,-1.912500
25505,18.690000,52.56,991.5,2025-11-28 17:00:00,0,normal,17,11,-2.210000,18.720851,7.670904,-0.030851,0.30,47.827500,14.717465,4.732500,0.3,993.037500,3.502522,-1.537500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31843,29.160000,85.16,979.3,2026-08-19 19:00:00,0,normal,19,8,-0.600000,29.077917,2.345245,0.082083,3.96,84.212083,12.570406,0.947917,0.9,978.950000,0.824621,0.350000
31844,28.720000,87.20,979.8,2026-08-19 20:00:00,0,normal,20,8,-0.440000,29.096250,2.340451,-0.376250,2.04,84.083333,12.521175,3.116667,0.5,979.008333,0.833493,0.791667
31845,28.350000,88.36,980.0,2026-08-19 21:00:00,0,normal,21,8,-0.370000,29.125417,2.325961,-0.775417,1.16,83.904167,12.423499,4.455833,0.2,979.058333,0.856137,0.941667
31846,27.960000,89.88,979.8,2026-08-19 22:00:00,0,normal,22,8,-0.390000,29.159583,2.301422,-1.199583,1.52,83.682083,12.259161,6.197917,-0.2,979.087500,0.869439,0.712500


In [42]:
print(test_ml.shape)
print(test_ml['anomaly_type'].value_counts())

(4282, 20)
anomaly_type
normal                        3864
temperature_drift              186
temperature_frozen              96
multivariate_inconsistency      75
temperature_spike               61
Name: count, dtype: int64
